# Stage 13C — frozen GLM and DeBERTa complement screen

Replay the fixed GLM control and compare it with a fixed DeBERTa complement on the same five prompt-held-out folds. This is exploratory development evidence, not an official Kaggle score or an out-of-fold replay of the full S07 ensemble.


## Private CPU inputs

| Input | Required kernel version |
|---|---:|
| Private Stage 13 GLM readout V3 | 368292220 |
| Frozen pretrained DeBERTa feature output | 358745836 |

Bind exactly these versions. Keep the notebook private, accelerator None and Internet off. The runner checks both cache digests and alignment before scoring. No row-level data is exported.


In [ ]:
from pathlib import Path
import hashlib
import json
import math
import subprocess
import sys

_sources = {'stage13c_deberta_complement.py': ('"""Stage 13C: one fixed DeBERTa readout and paired, fold-local GLM comparison."""\nfrom __future__ import annotations\n\nimport argparse\nfrom fractions import Fraction\nimport hashlib\nimport io\nimport json\nfrom pathlib import Path\nimport pickletools\nimport time\nimport warnings\nimport zipfile\n\nimport numpy as np\nfrom sklearn import __version__ as sklearn_version\nfrom sklearn.exceptions import ConvergenceWarning\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.preprocessing import StandardScaler\nfrom threadpoolctl import threadpool_info, threadpool_limits\n\nimport stage13b_balancing_stress as reference\n\n\nCOMPLETE = "COMPLETE_GLM_DEBERTA_COMPLEMENT_SCREEN"\nFAILED = "FAILED_GLM_DEBERTA_COMPLEMENT_SCREEN"\nSCOPE = "glm_deberta_paired_fold_local_development"\nDEBERTA_VERSION = 358745836\nDEBERTA_SHA = "11eed58762ad128f97457dde9cc4ab03c7932e1b4b04845ec6664c834b1b1dda"\nDEBERTA_BYTES = 99537639\nRAW_CONTROL_CM_SHA = "fa3ced3a896ce9389bf0427a42a30d8e7a30089c792cb92b8988953204b98747"\nUNIFORM_CONTROL_CM_SHA = "e81a80974d67dc60dc93faf3144dff56ab2055a327abcf7805dbb3556f364b0b"\nUNIFORM_CONTROL_BA = 0.4608252547849864\nSKLEARN_VERSION = "1.6.1"\nTHREADS = 2\nHEAD = dict(C=0.005, solver="lbfgs", max_iter=5000, tol=1e-4, fit_intercept=True, random_state=0)\nK, ROWS, FOLDS = reference.K, reference.ROWS, reference.FOLDS\nrequire = reference.require\n\n\ndef header(stream):\n    """Check the NPY metadata before array materialization."""\n    version = np.lib.format.read_magic(stream)\n    require(version in ((1, 0), (2, 0)), "unsupported_deberta_npy_version")\n    read = np.lib.format.read_array_header_1_0 if version == (1, 0) else np.lib.format.read_array_header_2_0\n    return read(stream, max_header_size=4096)\n\n\ndef checked_ids(blob, expected):\n    """Verify the frozen ID member without deserializing any object array."""\n    stream = io.BytesIO(blob)\n    shape, fortran, dtype = header(stream)\n    require(shape == expected.shape and not fortran, "deberta_ids_shape_or_order")\n    if not dtype.hasobject:\n        require(dtype.kind in "SU", "deberta_ids_numeric_or_invalid_dtype")\n        actual = np.load(io.BytesIO(blob), allow_pickle=False).astype(str)\n        mode = "numpy_nonobject"\n    else:\n        require(dtype == np.dtype(object), "deberta_ids_object_dtype")\n        payload = stream.read()\n        id_set = set(expected.tolist())\n        literals = []\n        # The byte hash is checked before opening the archive. genops parses\n        # bytecode, never imports symbols or executes a pickle instruction.\n        last = None\n        for n, (op, value, pos) in enumerate(pickletools.genops(payload), 1):\n            require(n <= 100000, "deberta_ids_opcode_limit")\n            last = (op.name, pos)\n            if isinstance(value, bytes):\n                try:\n                    value = value.decode("ascii")\n                except UnicodeDecodeError:\n                    continue\n            if isinstance(value, str) and value in id_set:\n                literals.append(value)\n        require(last == ("STOP", len(payload) - 1), "deberta_ids_pickle_trailing_data")\n        actual = np.asarray(literals, dtype=str)\n        mode = "object_npy_static_unique_ID_literals_no_unpickling"\n    require(np.array_equal(actual, expected), "deberta_ids_order_mismatch")\n    return mode\n\n\ndef load_features(path, ids):\n    with path.open("rb") as f:\n        require(path.stat().st_size == DEBERTA_BYTES, "deberta_cache_size_mismatch")\n        require(hashlib.file_digest(f, "sha256").hexdigest() == DEBERTA_SHA, "deberta_cache_hash_mismatch")\n        f.seek(0)\n        with zipfile.ZipFile(f) as z:\n            for key, limit in (("ids.npy", 2_000_000), ("layers.npy", 4096),\n                               ("emb.npy", ROWS * 9 * 1024 * 2 + 4096)):\n                require(z.namelist().count(key) == 1 and z.getinfo(key).file_size < limit,\n                        "deberta_cache_member_bounds")\n            mode = checked_ids(z.read("ids.npy"), ids)\n            with z.open("emb.npy") as member:\n                shape, _, dtype = header(member)\n                require(shape == (ROWS, 9, 1024) and dtype == np.dtype(np.float16), "deberta_emb_shape_or_dtype")\n            with z.open("layers.npy") as member:\n                shape, _, dtype = header(member)\n                require(shape == (9,) and dtype.kind in "iu", "deberta_layers_shape_or_dtype")\n        f.seek(0)\n        with np.load(f, allow_pickle=False) as z:\n            layers, emb = z["layers"], z["emb"]\n            require(layers.shape == (9,) and layers.dtype.kind in "iu" and len(np.unique(layers)) == 9,\n                    "deberta_layers_invalid")\n            require(all((layers == layer).sum() == 1 for layer in (24, 6)), "deberta_missing_fixed_layers")\n            require(emb.shape == (ROWS, 9, 1024) and emb.dtype == np.float16 and np.isfinite(emb).all(),\n                    "deberta_emb_validation")\n            x = np.ascontiguousarray(np.concatenate([emb[:, np.flatnonzero(layers == layer).item()]\n                                                     for layer in (24, 6)], axis=1), dtype=np.float64)\n    require(x.shape == (ROWS, 2048) and np.isfinite(x).all(), "deberta_selected_features")\n    return x, mode\n\n\ndef fold_masks(prompts, y, folds):\n    """Validate all fitting/scoring boundaries before fitting any head."""\n    require(prompts.shape == y.shape == folds.shape == (ROWS,), "complement_axis_shape")\n    masks = []\n    for f in range(1, FOLDS + 1):\n        val = folds == f\n        fit_prompts, val_prompts = np.unique(prompts[~val]), np.unique(prompts[val])\n        require(len(fit_prompts) == 120 and len(val_prompts) == 30\n                and len(np.intersect1d(fit_prompts, val_prompts)) == 0\n                and int(val.sum()) == (540 if f < 5 else 539), "complement_fold_exclusion")\n        require(all(np.array_equal(np.unique(y[m]), np.arange(K)) for m in (val, ~val)),\n                "complement_fold_class_coverage")\n        masks.append(val)\n    require(np.asarray(masks).sum(axis=0).tolist() == [1] * ROWS, "complement_fold_cover")\n    return masks\n\n\ndef fit_oof(x, y, masks):\n    require(x.ndim == 2 and len(x) == ROWS and x.dtype == np.float64 and np.isfinite(x).all(),\n            "complement_fit_features")\n    p = np.full((ROWS, K), np.nan, dtype=np.float64)\n    filled = np.zeros(ROWS, dtype=np.int8)\n    details = []\n    with threadpool_limits(limits=THREADS):\n        pools = [{"api": item["user_api"], "threads": item["num_threads"]} for item in threadpool_info()]\n        require(all(0 < pool["threads"] <= THREADS for pool in pools), "complement_thread_limit")\n        for fold, val in enumerate(masks, 1):\n            started = time.monotonic()\n            scaler = StandardScaler().fit(x[~val])\n            with warnings.catch_warnings():\n                warnings.simplefilter("error", ConvergenceWarning)\n                estimator = LogisticRegression(**HEAD).fit(scaler.transform(x[~val]), y[~val])\n            require(np.array_equal(estimator.classes_, np.arange(K)), "complement_estimator_class_axis")\n            niter = [int(n) for n in estimator.n_iter_]\n            require(len(niter) == 1 and 0 < niter[0] < HEAD["max_iter"], "complement_iteration_limit")\n            scores = estimator.predict_proba(scaler.transform(x[val]))\n            reference.validate_p(scores)\n            require(len(scores) == val.sum() and (filled[val] == 0).all(), "complement_duplicate_or_missing_rows")\n            p[val], filled[val] = scores, 1\n            details.append(dict(fold=fold, iterations=niter[0], fit_rows=int((~val).sum()),\n                                valid_rows=int(val.sum()), fit_prompts=120, valid_prompts=30,\n                                prompt_overlap=0, seconds=round(time.monotonic() - started, 6)))\n    require((filled == 1).all() and len(details) == FOLDS, "complement_oof_cover")\n    reference.validate_p(p)\n    return p, details, pools\n\n\ndef replay_control(p, y, masks, producer):\n    """Require the full prior confusion evidence before fitting anything."""\n    predicted = np.full(ROWS, -1, dtype=np.int64)\n    residuals = []\n    for mask in masks:\n        scores, residual = reference.evidence(p[mask], True)\n        predicted[mask] = scores.argmax(axis=1)\n        residuals.append(residual)\n    require((predicted >= 0).all(), "control_oof_cover")\n    raw = reference.confusion(y, reference.evidence(p, False)[0].argmax(axis=1))\n    uniform = reference.confusion(y, predicted)\n    require(raw.tolist() == producer["confusion_counts"]\n            and reference.digest_json(raw.tolist()) == RAW_CONTROL_CM_SHA\n            and abs(float(reference.recalls(raw).mean()) - reference.EXPECTED_RECALL) < 1e-15,\n            "complement_raw_control_confusion_mismatch")\n    require(reference.digest_json(uniform.tolist()) == UNIFORM_CONTROL_CM_SHA\n            and abs(float(reference.recalls(uniform).mean()) - UNIFORM_CONTROL_BA) < 1e-15,\n            "complement_uniform_control_confusion_mismatch")\n    return predicted, uniform, residuals\n\n\ndef paired_scores(glm, deb, prompts, y, masks, control, control_cm, residuals, folds):\n    """Apply the same frozen balancing to full, separate scoring folds."""\n    candidate = np.full(ROWS, -1, dtype=np.int64)\n    alone = np.full(ROWS, -1, dtype=np.int64)\n    rng = np.random.default_rng(reference.SEED)\n    bootstrap_support = np.zeros((reference.DRAWS, K), dtype=np.int64)\n    bootstrap_diff = np.zeros_like(bootstrap_support)\n    for f, mask in enumerate(masks):\n        pp = 0.75 * glm[mask] + 0.25 * deb[mask]\n        scores, candidate_residual = reference.evidence(pp, True)\n        candidate[mask] = scores.argmax(axis=1)\n        alone_scores, _ = reference.evidence(deb[mask], True)\n        alone[mask] = alone_scores.argmax(axis=1)\n        yt, gc, mix = y[mask], control[mask], candidate[mask]\n        cg, cc = reference.confusion(yt, gc), reference.confusion(yt, mix)\n        cgood, ggood = mix == yt, gc == yt\n        ordered = np.unique(prompts[mask])\n        weights = rng.multinomial(30, np.full(30, 1 / 30), size=reference.DRAWS)\n        support, diff = reference.bootstrap_delta(np.searchsorted(ordered, prompts[mask]), yt, cgood, ggood, weights)\n        bootstrap_support += support\n        bootstrap_diff += diff\n        folds[f].update(control_ba=float(reference.recalls(cg).mean()),\n                        candidate_ba=float(reference.recalls(cc).mean()),\n                        control_column_mass_error=residuals[f],\n                        candidate_column_mass_error=candidate_residual,\n                        corrected_by_candidate=int((cgood & ~ggood).sum()),\n                        broken_by_candidate=int((ggood & ~cgood).sum()))\n    require((bootstrap_support > 0).all(), "complement_bootstrap_missing_class")\n    require((candidate >= 0).all() and (alone >= 0).all(), "complement_predictions_incomplete")\n    cand_cm, alone_cm = reference.confusion(y, candidate), reference.confusion(y, alone)\n    raw_deb_cm = reference.confusion(y, reference.evidence(deb, False)[0].argmax(axis=1))\n    controls, candidates = reference.recalls(control_cm), reference.recalls(cand_cm)\n    supports = control_cm.sum(axis=1)\n    require(np.array_equal(supports, cand_cm.sum(axis=1)), "complement_support_mismatch")\n    exact = [Fraction(int(cand_cm[i, i]) - int(control_cm[i, i]), int(supports[i])) * 100 for i in range(K)]\n    delta = sum(exact) / K\n    ci = np.percentile((bootstrap_diff / bootstrap_support).mean(axis=1) * 100, [2.5, 97.5]).tolist()\n    corrected = int(((candidate == y) & (control != y)).sum())\n    broken = int(((control == y) & (candidate != y)).sum())\n    changed = int((candidate != control).sum())\n    require(corrected + broken <= changed\n            and corrected - broken == int(cand_cm.trace() - control_cm.trace())\n            and corrected == sum(f["corrected_by_candidate"] for f in folds)\n            and broken == sum(f["broken_by_candidate"] for f in folds), "complement_paired_count_mismatch")\n    criteria = dict(pooled_gain_at_least_half_pp=delta >= Fraction(1, 2),\n                    conditional_ci_lower_positive=ci[0] > 0,\n                    every_class_recall_loss_le_five_pp=min(exact) >= -5)\n    return dict(control_ba=float(controls.mean()), candidate_ba=float(candidates.mean()),\n                deberta_alone_ba=float(reference.recalls(alone_cm).mean()),\n                deberta_alone_raw_ba=float(reference.recalls(raw_deb_cm).mean()),\n                delta_pp=float(delta), delta_pp_exact=str(delta), conditional_delta_ci95_pp=ci,\n                class_supports=supports.tolist(), class_delta_pp=[float(v) for v in exact],\n                control_class_recalls=controls.tolist(), candidate_class_recalls=candidates.tolist(),\n                control_confusion_counts=control_cm.tolist(), candidate_confusion_counts=cand_cm.tolist(),\n                deberta_alone_confusion_counts=alone_cm.tolist(),\n                deberta_alone_raw_confusion_counts=raw_deb_cm.tolist(),\n                corrected_by_candidate=corrected, broken_by_candidate=broken, changed_by_candidate=changed,\n                decision_criteria=criteria, passes_proxy_screen=all(criteria.values()))\n\n\ndef run(input_root, output_root):\n    started = time.monotonic()\n    root, out = Path(input_root), Path(output_root)\n    report = dict(schema_version=1, status=FAILED, scope=SCOPE, fit_attempted=False, fit_performed=False,\n                  raw_rows_or_predictions_exported=False, official_score_estimated=False,\n                  full_s07_oof_admitted=False, expected_input_version_ids=[reference.SOURCE_VERSION, DEBERTA_VERSION],\n                  heldout_cache_sha256=reference.CACHE_SHA, deberta_cache_sha256=DEBERTA_SHA)\n    writable = False\n    try:\n        require(not out.exists() or (out.is_dir() and not any(out.iterdir())), "fresh_output_directory")\n        out.mkdir(parents=True, exist_ok=True)\n        writable = True\n        require(sklearn_version == SKLEARN_VERSION, "complement_sklearn_version")\n        ids, prompts, y, folds, classes, glm, producer = reference.cache_inputs(root)\n        masks = fold_masks(prompts, y, folds)\n        control, cm, residuals = replay_control(glm, y, masks, producer)\n        report["control_replayed"] = True\n        features, id_mode = load_features(reference.one(root, "enc_debv3l2_train.npz"), ids)\n        report["fit_attempted"] = True\n        deb, details, pools = fit_oof(features, y, masks)\n        report["fit_performed"] = True\n        scores = paired_scores(glm, deb, prompts, y, masks, control, cm, residuals, details)\n        require(hashlib.sha256(glm.astype("<f8", copy=False).tobytes()).hexdigest() == reference.PROB_SHA,\n                "complement_glm_mutated")\n        report.update(status=COMPLETE, rows=ROWS, prompts=reference.PROMPTS, classes=classes.tolist(), folds=details,\n                      estimator=HEAD, thread_pools=pools, feature_layers=[24, 6], feature_dtype="float16_cached",\n                      deberta_id_verification=id_mode, sklearn_version=sklearn_version, numpy_version=np.__version__,\n                      control_raw_confusion_sha256=RAW_CONTROL_CM_SHA, control_uniform_confusion_sha256=UNIFORM_CONTROL_CM_SHA,\n                      mixture_weights={"glm": 0.75, "deberta": 0.25},\n                      balancing={"alpha": 1, "iterations": 100, "minimum_batch_rows": 18, "fold_local": True},\n                      bootstrap={"draws": reference.DRAWS, "seed": reference.SEED,\n                                 "scope": "conditional stratified paired prompt resampling on fixed predictions"},\n                      **scores)\n    except reference.GateError as error:\n        report.update(failure_code=str(error))\n    except Exception as error:\n        report.update(failure_code="runtime_failure", failure_type=type(error).__name__)\n    report["seconds"] = round(time.monotonic() - started, 6)\n    if writable:\n        try:\n            tmp = out / ".complement_report.tmp"\n            tmp.write_text(json.dumps(report, sort_keys=True, indent=2, allow_nan=False) + "\\n", encoding="utf-8")\n            tmp.replace(out / "complement_report.json")\n        except Exception as error:\n            report.update(status=FAILED, failure_code="report_write_failure", failure_type=type(error).__name__)\n    return report\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--input-root", default="/kaggle/input")\n    parser.add_argument("--output-root", default="/kaggle/working/stage13c_deberta_complement")\n    args = parser.parse_args()\n    result = run(args.input_root, args.output_root)\n    print("STAGE13C_GLM_DEBERTA_JSON_BEGIN", flush=True)\n    print(json.dumps(result, sort_keys=True, allow_nan=False), flush=True)\n    print("STAGE13C_GLM_DEBERTA_JSON_END", flush=True)\n    return 0 if result["status"] == COMPLETE else 2\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'b9b6f523b8a6b1f9f1dae556c863f7fee34f56fddfb27adb0946021e1f17181e'), 'stage13b_balancing_stress.py': ('"""Run one predeclared balancing diagnostic on the private GLM OOF cache."""\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\nimport time\n\nimport numpy as np\n\n\nCOMPLETE = "COMPLETE_GLM_SINKHORN_STRESS_ONLY"\nSCOPE = "glm_only_paired_fold_local_balancing_stress"\nSOURCE_VERSION = 368292220\nCACHE_SHA = "565df6f8e652a9850ff11cf4e8b6a81fcccbc7db593186ff7f53358e7190bd16"\nPROB_SHA = "41d80bc9350ab44d5aabe02b24e0174881a307bb1eb962f04196682e7faa94fe"\nID_SHA = "707edf645b0f09bba8f0983b5bc9f522595c95e8ca5b763b38babeb38200368d"\nLABEL_SHA = "088b14188775e41a314f269da3af74f96144a56caa242ccbeacc9e3c817a7e08"\nFOLD_SHA = "a55b035686cb7b31f6fbe852ac990668709aec4b913c5d0928d89d843bdbfe62"\nEXPECTED_RECALL = 61547 / 134100\nK, ROWS, PROMPTS, FOLDS = 18, 2699, 150, 5\nSCENARIOS = ("natural_full", "matched_balanced", "skew_a", "skew_b")\nDRAWS, SEED = 5000, 20261111\nATOL = 1e-12\n\n\nclass GateError(Exception):\n    pass\n\n\ndef require(condition, code):\n    if not bool(condition):\n        raise GateError(code)\n\n\ndef digest_json(obj):\n    return hashlib.sha256(json.dumps(obj, sort_keys=True, separators=(",", ":")).encode()).hexdigest()\n\n\ndef salted_hash(salt, value):\n    return hashlib.sha256((salt + "\\0" + str(value)).encode("utf-8")).digest()\n\n\ndef one(root, name):\n    matches = [p for p in root.rglob(name) if p.is_file()]\n    require(len(matches) == 1, "unique_" + name.replace(".", "_"))\n    return matches[0]\n\n\ndef validate_p(p):\n    require(p.ndim == 2 and p.shape[1] == K and p.dtype == np.float64 and len(p) > 0,\n            "probability_shape_or_dtype")\n    require(np.isfinite(p).all() and (p > 0).all() and (p <= 1).all(), "invalid_probability_values")\n    require(np.max(np.abs(p.sum(axis=1) - 1.0)) <= ATOL, "probability_row_sums")\n\n\ndef evidence(p, uniform):\n    """Use the source\'s fixed 100 updates and branch threshold on existing P."""\n    validate_p(p)\n    if not uniform or len(p) < K:\n        return np.log(p + 1e-12), None\n    q = p.copy()\n    with np.errstate(divide="raise", invalid="raise", over="raise"):\n        for _ in range(100):\n            q *= (len(q) / 18) / q.sum(0, keepdims=True)\n            q /= q.sum(1, keepdims=True)\n    validate_p(q)\n    residual = float(np.max(np.abs(q.sum(axis=0) - len(q) / K)))\n    require(np.isfinite(residual), "nonfinite_column_mass_error")\n    return np.log(q + 1e-12), residual\n\n\ndef cache_inputs(root):\n    cache = one(root, "heldout_probabilities.npz")\n    report = one(root, "readout_report.json")\n    require(cache.parent == report.parent, "cache_report_parent_mismatch")\n    with cache.open("rb") as f:\n        require(hashlib.file_digest(f, "sha256").hexdigest() == CACHE_SHA, "cache_byte_hash_mismatch")\n    r = json.loads(report.read_text(encoding="utf-8"))\n    require(r.get("status") == "COMPLETE_GLM_PLAIN_GROUPED_DEVELOPMENT_REFERENCE"\n            and r.get("scope") == "glm_only_prompt_heldout_development"\n            and r.get("heldout_cache_sha256") == CACHE_SHA\n            and r.get("probability_values_sha256") == PROB_SHA\n            and r.get("admission_checks_passed") == 7, "producer_report_contract")\n    for flag in ("fit_performed", "heldout_predictions_produced", "heldout_cache_readback_passed"):\n        require(r.get(flag) is True, "producer_report_missing_required_flag")\n    for flag in ("balancing_performed", "official_score_estimated", "full_s07_oof_admitted", "raw_rows_or_predictions_exported"):\n        require(r.get(flag) is False, "producer_report_scope")\n    keys = ("ids", "prompt_ids", "targets", "folds", "classes", "probabilities")\n    with np.load(cache, allow_pickle=False) as z:\n        require(len(z.files) == len(keys) and set(z.files) == set(keys), "cache_schema")\n        ids, prompts, targets, folds, classes, p = (z[k] for k in keys)\n    require(ids.shape == prompts.shape == targets.shape == folds.shape == (ROWS,), "cache_row_axes")\n    require(ids.dtype.kind == prompts.dtype.kind == classes.dtype.kind == "U"\n            and targets.dtype == folds.dtype == np.int64 and classes.shape == (K,), "cache_axis_dtypes")\n    require(len(np.unique(ids)) == ROWS and len(np.unique(prompts)) == PROMPTS, "cache_unique_axes")\n    require(digest_json(ids.tolist()) == ID_SHA and digest_json(classes.tolist()) == LABEL_SHA, "cache_axis_hashes")\n    require(np.array_equal(np.unique(targets), np.arange(K)) and np.array_equal(np.unique(folds), np.arange(1, 6)),\n            "cache_class_or_fold_indices")\n    require(np.bincount(targets).tolist() == [149] + [150] * 17, "original_class_supports")\n    validate_p(p)\n    require(len(p) == ROWS and hashlib.sha256(p.astype("<f8", copy=False).tobytes()).hexdigest() == PROB_SHA,\n            "cache_probability_hash")\n    mapping = {}\n    for group, fold in zip(prompts, folds):\n        require(mapping.setdefault(str(group), int(fold)) == int(fold), "prompt_crosses_fold")\n    require(digest_json(mapping) == FOLD_SHA and r.get("canonical_prompt_fold_mapping_sha256") == FOLD_SHA,\n            "canonical_fold_map_hash")\n    require(r.get("classes") == classes.tolist() and len(r.get("folds", [])) == FOLDS, "producer_fold_axis")\n    for f, item in enumerate(r["folds"], 1):\n        n = 540 if f < 5 else 539\n        mask = folds == f\n        require(item.get("fold") == f and int(mask.sum()) == n and len(np.unique(prompts[mask])) == 30\n                and [item.get(k) for k in ("fit_rows", "valid_rows", "fit_prompts", "valid_prompts", "prompt_overlap")]\n                == [ROWS - n, n, 120, 30, 0], "producer_fold_exclusion_contract")\n    return ids, prompts, targets, folds, classes, p, r\n\n\ndef mask_indices(ids, targets, classes, indices):\n    """Rank once per class. Use matched subsets for all paired scoring arms."""\n    ranked = sorted(range(K), key=lambda c: (salted_hash("stage13b-class-v1", classes[c]), str(classes[c]).encode("utf-8")))\n    a_major = set(ranked[:9])\n    buckets = [sorted(indices[targets[indices] == c].tolist(),\n                      key=lambda i: (salted_hash("stage13b-row-v1", ids[i]), str(ids[i]).encode("utf-8"))) for c in range(K)]\n    require(all(len(rows) >= 27 for rows in buckets), "insufficient_mask_class_support")\n    masks = {"natural_full": indices}\n    for name in SCENARIOS[1:]:\n        counts = [20] * K if name == "matched_balanced" else [\n            27 if ((c in a_major) == (name == "skew_a")) else 13 for c in range(K)]\n        chosen = np.asarray(sorted(i for c, rows in enumerate(buckets) for i in rows[:counts[c]]), dtype=np.int64)\n        require(len(chosen) == len(np.unique(chosen)) == 360 and np.bincount(targets[chosen], minlength=K).tolist() == counts,\n                "matched_mask_supports")\n        masks[name] = chosen\n    return masks\n\n\ndef confusion(y, pred):\n    return np.bincount(y * K + pred, minlength=K*K).reshape(K, K)\n\n\ndef recalls(counts):\n    totals = counts.sum(axis=1)\n    require((totals > 0).all(), "class_missing_from_score")\n    return counts.diagonal() / totals\n\n\ndef bootstrap_delta(prompt_index, targets, correct_off, correct_uniform, weights):\n    supports = np.zeros((30, K), dtype=np.int64)\n    differences = np.zeros_like(supports)\n    np.add.at(supports, (prompt_index, targets), 1)\n    np.add.at(differences, (prompt_index, targets), correct_off.astype(np.int64) - correct_uniform.astype(np.int64))\n    return weights @ supports, weights @ differences\n\n\ndef diagnostics(ids, prompts, targets, folds, classes, p, producer):\n    # Freeze the shared paired resampling before applying either arm.\n    rng = np.random.default_rng(SEED)\n    weights = [rng.multinomial(30, np.full(30, 1/30), size=DRAWS) for _ in range(FOLDS)]\n    stores = {n: {"off": np.zeros((K, K), dtype=np.int64), "uniform": np.zeros((K, K), dtype=np.int64),\n                  "support_draws": np.zeros((DRAWS, K), dtype=np.int64), "delta_draws": np.zeros((DRAWS, K), dtype=np.int64),\n                  "fixed": 0, "broken": 0, "changed": 0, "batches": []} for n in SCENARIOS}\n    digest_before = hashlib.sha256(p.astype("<f8", copy=False).tobytes()).hexdigest()\n    for f in range(1, 6):\n        indices = np.flatnonzero(folds == f)\n        ordered_groups = np.unique(prompts[indices])\n        require(len(ordered_groups) == 30, "scoring_batch_prompt_count")\n        for name, selected in mask_indices(ids, targets, classes, indices).items():\n            require(len(np.unique(prompts[selected])) == 30 and (folds[selected] == f).all(), "masked_batch_fold_or_prompt_count")\n            y, pp = targets[selected], p[selected]\n            log_off, _ = evidence(pp, False)\n            log_uniform, residual = evidence(pp, True)\n            pred_off, pred_uniform = log_off.argmax(1), log_uniform.argmax(1)\n            off, uniform = confusion(y, pred_off), confusion(y, pred_uniform)\n            co, cu = pred_off == y, pred_uniform == y\n            fixed, broken = int((co & ~cu).sum()), int((cu & ~co).sum())\n            changed = int((pred_off != pred_uniform).sum())\n            require(fixed + broken <= changed and int(co.sum() - cu.sum()) == fixed - broken, "paired_counts")\n            b = stores[name]\n            b["off"] += off\n            b["uniform"] += uniform\n            b["fixed"] += fixed\n            b["broken"] += broken\n            b["changed"] += changed\n            b["batches"].append(dict(fold=f, rows=len(y), prompts=30, class_supports=off.sum(1).tolist(),\n                                     off_ba=float(recalls(off).mean()), uniform_ba=float(recalls(uniform).mean()),\n                                     uniform_max_column_mass_error=residual))\n            ds, dd = bootstrap_delta(np.searchsorted(ordered_groups, prompts[selected]), y, co, cu, weights[f-1])\n            b["support_draws"] += ds\n            b["delta_draws"] += dd\n    natural = stores["natural_full"]["off"]\n    require(np.array_equal(natural, np.asarray(producer["confusion_counts"]))\n            and abs(float(recalls(natural).mean()) - EXPECTED_RECALL) <= 1e-15\n            and abs(float(producer["pooled_balanced_accuracy"]) - EXPECTED_RECALL) <= 1e-15, "original_plain_reference_replay")\n    require(hashlib.sha256(p.astype("<f8", copy=False).tobytes()).hexdigest() == digest_before, "mutated_upstream_probabilities")\n    results = {}\n    for name, b in stores.items():\n        off, uniform = b["off"], b["uniform"]\n        ro, ru = recalls(off), recalls(uniform)\n        require(np.array_equal(off.sum(1), uniform.sum(1)) and (b["support_draws"] > 0).all(), "paired_support_or_bootstrap_coverage")\n        intervals = np.percentile((b["delta_draws"] / b["support_draws"]).mean(axis=1) * 100, [2.5, 97.5])\n        results[name] = dict(rows=int(off.sum()), batches=b["batches"], off_ba=float(ro.mean()), uniform_ba=float(ru.mean()),\n                             delta_pp=float((ro.mean() - ru.mean()) * 100), conditional_delta_ci95_pp=intervals.tolist(),\n                             class_supports=off.sum(1).tolist(), class_delta_pp=((ro - ru) * 100).tolist(),\n                             off_class_recalls=ro.tolist(), uniform_class_recalls=ru.tolist(),\n                             off_confusion_counts=off.tolist(), uniform_confusion_counts=uniform.tolist(),\n                             corrected_by_off=b["fixed"], broken_by_off=b["broken"], changed_by_off=b["changed"])\n    require([results[n]["rows"] for n in SCENARIOS] == [2699, 1800, 1800, 1800], "scenario_row_counts")\n    criteria = dict(\n        both_skew_point_gains_ge_half_pp=all(results[n]["delta_pp"] >= 0.5 for n in ("skew_a", "skew_b")),\n        both_skew_conditional_ci_lowers_positive=all(results[n]["conditional_delta_ci95_pp"][0] > 0 for n in ("skew_a", "skew_b")),\n        natural_and_matched_losses_le_half_pp=all(results[n]["delta_pp"] >= -0.5 for n in ("natural_full", "matched_balanced")),\n        every_class_recall_loss_le_five_pp=all(min(b["class_delta_pp"]) >= -5.0 - 1e-12 for b in results.values()),\n    )\n    return results, criteria\n\n\ndef run(input_root, output_root):\n    t = time.monotonic()\n    root, out = Path(input_root), Path(output_root)\n    report = dict(schema_version=1, status="FAILED_GLM_SINKHORN_STRESS", scope=SCOPE, source_version_id=SOURCE_VERSION,\n                  heldout_cache_sha256=CACHE_SHA, fit_performed=False, official_score_estimated=False,\n                  full_s07_oof_admitted=False, raw_rows_or_predictions_exported=False)\n    writable = False\n    try:\n        require(not out.exists() or (out.is_dir() and not any(out.iterdir())), "fresh_output_directory")\n        out.mkdir(parents=True, exist_ok=True)\n        writable = True\n        ids, groups, targets, folds, classes, p, prior = cache_inputs(root)\n        results, criteria = diagnostics(ids, groups, targets, folds, classes, p, prior)\n        report.update(status=COMPLETE, scenarios=results, classes=classes.tolist(), decision_criteria=criteria,\n                      passes_proxy_screen=all(criteria.values()), probabilities_unchanged=True,\n                      original_plain_reference_replayed=True, actual_folds=5, prompts=150,\n                      bootstrap=dict(draws=DRAWS, seed=SEED, scope="fixed-prediction stratified prompt resampling"),\n                      control=dict(alpha=1, iterations=100, minimum_batch_rows=18), candidate=dict(alpha=0),\n                      delta_direction="off_minus_uniform", numpy_version=np.__version__)\n    except GateError as error:\n        report.update(failure_code=str(error))\n    except Exception as error:\n        report.update(failure_code="runtime_failure", failure_type=type(error).__name__)\n    report["seconds"] = round(time.monotonic() - t, 6)\n    if writable:\n        try:\n            tmp = out / ".balancing_stress_report.tmp"\n            tmp.write_text(json.dumps(report, sort_keys=True, indent=2, allow_nan=False) + "\\n", encoding="utf-8")\n            tmp.replace(out / "balancing_stress_report.json")\n        except Exception as error:\n            report.update(status="FAILED_GLM_SINKHORN_STRESS", failure_code="report_write_failure", failure_type=type(error).__name__)\n    return report\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--input-root", default="/kaggle/input")\n    parser.add_argument("--output-root", default="/kaggle/working/stage13b_balancing_stress")\n    args = parser.parse_args()\n    result = run(args.input_root, args.output_root)\n    print("STAGE13B_GLM_STRESS_JSON_BEGIN", flush=True)\n    print(json.dumps(result, sort_keys=True, allow_nan=False), flush=True)\n    print("STAGE13B_GLM_STRESS_JSON_END", flush=True)\n    return 0 if result["status"] == COMPLETE else 2\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'f4981f385074f60ee034b979d4aa6672c7d28eaa408c374d5caf55a7c446fbd9')}
_source_dir = Path('/kaggle/working/_stage13c_deberta_complement_source')
_source_dir.mkdir(parents=True, exist_ok=True)
for _name, (_source, _expected) in _sources.items():
    _source_bytes = _source.encode('utf-8')
    if hashlib.sha256(_source_bytes).hexdigest() != _expected:
        raise RuntimeError('Stage13C embedded source hash mismatch')
    _path = _source_dir / _name
    _path.write_bytes(_source_bytes)
    if hashlib.sha256(_path.read_bytes()).hexdigest() != _expected:
        raise RuntimeError('Stage13C written source hash mismatch')
_runner_path = _source_dir / 'stage13c_deberta_complement.py'
print('STAGE13C_SOURCE_CHECKS_PASS', _sources[_runner_path.name][1])


## Run the frozen comparison

The job stops if an input, fold, control replay, or numerical check fails. The runner saves aggregate evidence only.


In [ ]:
subprocess.run([sys.executable, '-B', str(_runner_path), '--input-root', '/kaggle/input', '--output-root', '/kaggle/working/stage13c_deberta_complement'], check=True, timeout=600)


## Verify the saved aggregate report

Check the report scope, inputs, control replay and score arithmetic. Display only the aggregate scores and the proxy-screen decision.


In [ ]:
_result_dir = Path('/kaggle/working/stage13c_deberta_complement')
_report = json.loads((_result_dir / 'complement_report.json').read_text(encoding='utf-8'))
def _require(condition, code):
    if not condition:
        raise RuntimeError('Stage13C saved report check failed: ' + code)
_require(type(_report) is dict, 'report_type')
_require(_report.get('status') == 'COMPLETE_GLM_DEBERTA_COMPLEMENT_SCREEN', 'status')
_require(_report.get('scope') == 'glm_deberta_paired_fold_local_development', 'scope')
_require(type(_report.get('rows')) is int and _report['rows'] == 2699, 'rows')
_require(type(_report.get('prompts')) is int and _report['prompts'] == 150, 'prompts')
_classes = _report.get('classes')
_require(type(_classes) is list and len(_classes) == 18 and
         all(type(_label) is str and _label for _label in _classes) and len(set(_classes)) == 18, 'classes')
_folds = _report.get('folds')
_require(type(_folds) is list and len(_folds) == 5 and all(type(_fold) is dict for _fold in _folds), 'folds')
_require(_report.get('expected_input_version_ids') == [368292220, 358745836], 'expected_input_version_ids')
_require(_report.get('heldout_cache_sha256') == '565df6f8e652a9850ff11cf4e8b6a81fcccbc7db593186ff7f53358e7190bd16', 'heldout_cache_sha256')
_require(_report.get('deberta_cache_sha256') == '11eed58762ad128f97457dde9cc4ab03c7932e1b4b04845ec6664c834b1b1dda', 'deberta_cache_sha256')
for _flag in ('fit_performed', 'control_replayed'):
    _require(_report.get(_flag) is True, _flag)
for _flag in ('official_score_estimated', 'full_s07_oof_admitted', 'raw_rows_or_predictions_exported'):
    _require(_report.get(_flag) is False, _flag)
_require(type(_report.get('passes_proxy_screen')) is bool, 'passes_proxy_screen')
_scores = {}
for _key in ('control_ba', 'candidate_ba', 'deberta_alone_ba'):
    _score = _report.get(_key)
    _require(type(_score) in (int, float) and 0 <= _score <= 1 and math.isfinite(_score), _key)
    _scores[_key] = _score
_require(math.isclose(_scores['control_ba'], 0.4608252547849864, abs_tol=1e-12, rel_tol=0), 'control_replay')
_delta = _report.get('delta_pp')
_require(type(_delta) in (int, float) and -100 <= _delta <= 100 and math.isfinite(_delta), 'delta_pp')
_require(math.isclose(_delta, (_scores['candidate_ba'] - _scores['control_ba']) * 100,
                      abs_tol=1e-10, rel_tol=0), 'delta_arithmetic')
_ci = _report.get('conditional_delta_ci95_pp')
_require(type(_ci) is list and len(_ci) == 2 and
         all(type(_bound) in (int, float) and -100 <= _bound <= 100 and math.isfinite(_bound) for _bound in _ci)
         and _ci[0] <= _ci[1], 'conditional_delta_ci95_pp')
_scores.update(delta_pp=_delta, conditional_delta_ci95_pp=_ci, passes_proxy_screen=_report['passes_proxy_screen'])
print('STAGE13C_SAVED_REPORT_READBACK_PASS', json.dumps(_scores, sort_keys=True, allow_nan=False))
